**Note for reviewers.** This notebook also contains analyses that are not reported in the paper: C2-C4 matched-player swap comparisons. The reported player interventions come from targetted_analysis (identity.json). Those sections are kept only so the notebook runs as one unit, and their outputs should not be read as paper results.

**New benchmark analysis (2026-10-05).** Uses the available 4L/2H/K16 checkpoints with weight decay **0.1**, seeds 100–104. This is a new exploratory analysis, not reproduction of SEL-4L wd0.01. References/conflict definitions are frozen historical inputs, verified by hash. All model-dependent outputs are recomputed. Validation match splits support the stated comparisons; prior test access prevents a fresh confirmatory claim. Population and intervention scope apply to each reported result. See `config/analysis_plan.json` and `SURGICAL_FIXES.md`.

# MB04 — Input Counterfactuals: Identity, Context and their Interaction

**Track B · Mechanistic interpretability of SEL-4L · Identity × Context project**

| Item | Content |
|---|---|
| **Question** | Does changing only the player (or team, or role) change SEL-4L's output; is the change in the direction of that player's habits; and does context change how much identity matters? |
| **Experiments** | E5 (confirmatory C2), E6 (C3), E7, E8 (C4) |
| **Population** | P-player, validation. Development and plots use both halves; C2–C4 are decided on the estimation half. Index-0 rows serve as a control. |
| **Outputs** | `results/MB04_interventions.parquet`, `results/MB04_factorial.parquet`, `results/MB04_summary.json`, `figures/MB04_*` |
| **Test data** | none |



**Intervention and population scope.** Player swaps change only the player factor while recorded team, role, competition and history stay fixed. These are factor-sensitivity interventions, not realistic player substitutions. Coherence is only a limited post hoc proxy. C2–C4 apply to dedicated-player conflict rows; index-0 rows have no dedicated player vector, but other features can carry identity information. Read the newly executed C2 results below; historical wd0.01 validation/test estimates are not results of this benchmark.

In [1]:
import os
import sys
import json
import time
import importlib
import importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def find_root(start):
    """`start` (or one of its ancestors) holding both project folders directly, else None."""
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "model_training_evaluation").is_dir() and (candidate / "mi_analysis").is_dir():
            return candidate
    return None


def search_root(bases, max_depth=6):
    """Breadth-first search under each base (e.g. a mounted Drive) for the project root."""
    for base in bases:
        base = Path(base)
        if not base.is_dir():
            continue
        found = find_root(base)
        if found is not None:
            return found
        frontier = [(base, 0)]
        while frontier:
            folder, depth = frontier.pop(0)
            if depth >= max_depth:
                continue
            try:
                children = [c for c in folder.iterdir() if c.is_dir() and not c.name.startswith(".")]
            except PermissionError:
                continue
            for child in children:
                if child.name == "model_training_evaluation" and (child.parent / "mi_analysis").is_dir():
                    return child.parent
                frontier.append((child, depth + 1))
    return None


if "SOCCER_MI_ROOT" in os.environ:
    ROOT = Path(os.environ["SOCCER_MI_ROOT"])
elif IN_COLAB:
    ROOT = search_root(["/content", "/content/drive/MyDrive", "/content/drive/My Drive"])
    if ROOT is None:
        raise RuntimeError(
            "BLOCKED: could not find a folder holding both model_training_evaluation/ and mi_analysis/ "
            "under /content or the mounted Drive. Upload or sync the project there, or set "
            "SOCCER_MI_ROOT to its exact path in a cell ABOVE this one, for example: "
            "import os; os.environ['SOCCER_MI_ROOT'] = '/content/drive/MyDrive/.../soccer_mi'")
else:
    ROOT = find_root(Path.cwd())
    if ROOT is None:
        raise RuntimeError(f"BLOCKED: no ancestor of {Path.cwd()} holds both model_training_evaluation/ "
                           "and mi_analysis/. Run this notebook from inside the project, or set SOCCER_MI_ROOT.")
TRACK_A_DIR = ROOT / "model_training_evaluation"
MI_DIR = Path(os.environ.get("MI_ANALYSIS_DIR", ROOT / "mi_analysis")).resolve()
assert (TRACK_A_DIR / "src" / "data.py").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'src' / 'data.py'} not found -- ROOT resolved to {ROOT}, which is wrong. "
    "Set SOCCER_MI_ROOT explicitly.")
assert (TRACK_A_DIR / "artifacts" / "events_prepared.parquet").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'artifacts' / 'events_prepared.parquet'} not found -- ROOT is wrong.")
RUN_MODE = os.environ.get("MI_RUN_MODE", "full")
assert RUN_MODE in ("full", "smoke")

for folder in [MI_DIR, MI_DIR / "src", MI_DIR / "notebooks", MI_DIR / "tests"]:
    folder.mkdir(parents=True, exist_ok=True)
(MI_DIR / "src" / "__init__.py").touch()
os.chdir(MI_DIR)

sys.path[:] = [entry for entry in sys.path if entry not in ("", ".", str(MI_DIR))]
if str(TRACK_A_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_A_DIR))
spec = importlib.util.spec_from_file_location("mi_src", MI_DIR / "src" / "__init__.py",
                                              submodule_search_locations=[str(MI_DIR / "src")])
mi_src = importlib.util.module_from_spec(spec)
sys.modules["mi_src"] = mi_src
spec.loader.exec_module(mi_src)

print("Colab:", IN_COLAB, "| RUN_MODE:", RUN_MODE)
print("ROOT:", ROOT)
print("Track A (read only):", TRACK_A_DIR, "| exists:", TRACK_A_DIR.exists())
print("Track B:", MI_DIR)

Colab: False | RUN_MODE: full
ROOT: /home/user/code/soccer_mi
Track A (read only): /home/user/code/soccer_mi/model_training_evaluation | exists: True
Track B: /home/user/code/soccer_mi/.mi_analysis_rerun


In [2]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import src.data as data
import src.metrics as metrics
import src.training as training
import mi_src.paths as paths
import mi_src.fingerprint as fingerprint
import mi_src.conflict as conflict
import mi_src.mi_setup as mi_setup
import mi_src.hooks as hooks
import mi_src.interventions as interventions
import mi_src.mi_metrics as mi_metrics
for module in [data, metrics, training, paths, fingerprint, conflict, mi_setup, hooks, interventions, mi_metrics]:
    importlib.reload(module)

if getattr(paths, "SCHEMA_VERSION", 1) < 2:
    raise RuntimeError("BLOCKED: mi_analysis/src/paths.py is older than this notebook expects. "
                       "Re-run MB01 section 0 (its three %%writefile cells), then restart the kernel.")
paths.check_modules(**{name: module for name, module in
                       [("paths", paths), ("fingerprint", fingerprint), ("mi_setup", mi_setup),
                        ("patching", globals().get("patching")), ("circuits", globals().get("circuits"))]
                       if module is not None})

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

OUT = paths.output_dir(MI_DIR, RUN_MODE)
SETTINGS = paths.RUN_SETTINGS[RUN_MODE]
TRACK = paths.track_a_paths(TRACK_A_DIR)
MANIFEST = OUT / "manifests" / "sha_manifest.json"
REGISTRY = OUT / "manifests" / "experiment_registry.jsonl"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(int(os.environ.get("MI_CPU_THREADS", "4")))
if RUN_MODE == "full" and DEVICE != "cuda":
    print("Note: no GPU found. The full run works on CPU but is much slower.")
training.configure_determinism()
training.set_seed(0)
ACTIONS = list(data.ACTION_NAMES)
print("Device:", DEVICE, "| outputs:", OUT)
print("Settings:", SETTINGS)

Note: no GPU found. The full run works on CPU but is much slower.


Device: cpu | outputs: /home/user/code/soccer_mi/.mi_analysis_rerun
Settings: {'reference_epochs': 25, 'batch_windows': 256, 'n_bootstrap': 2000, 'n_draws': 5, 'n_permutations': 1000, 'n_random_circuits': 100, 'probe_train_events': 20000, 'max_events': 100000, 'probe_seeds': 3, 'scrub_draws': 5}


In [3]:
verified = fingerprint.verify_files(MANIFEST)
prereg = mi_setup.load_json(OUT / "config" / "preregistration.json")
mi_config = mi_setup.load_json(OUT / "config" / "mi_config.json")
IDENTITY = mi_config["identity_variable"]
PRIMARY_SEED = int(prereg["model"]["primary_seed"])


def run_dir(seed):
    """Run directory of one SEL-4L seed, re-rooted onto this checkout."""
    return paths.rebase_run_dir(prereg["model"]["run_dirs"][str(seed)], TRACK["root"])

SEEDS = [int(seed) for seed in prereg["model"]["run_dirs"]]
N_BOOT = SETTINGS["n_bootstrap"]
BATCH = SETTINGS["batch_windows"]

events = pd.read_parquet(TRACK["events"])
bundle = mi_setup.build_bundle(events, TRACK, DEVICE)
val_view = bundle["views"]["val"]
targets = bundle["targets"]
POP = OUT / "artifacts" / "populations"
val_conflicts = pd.read_parquet(POP / "val_conflicts.parquet")
controls = pd.read_parquet(POP / "matched_controls.parquet")
MODEL = mi_setup.load_model(run_dir(PRIMARY_SEED), bundle)
N_LAYERS = len(MODEL.blocks)
N_HEADS = MODEL.blocks[0].attention.n_heads
print(f"verified files: {len(verified)} | primary seed {PRIMARY_SEED} | identity variable: {IDENTITY}")
print(val_conflicts.groupby(["population", "half"]).size().unstack().to_string())

partners = pd.read_parquet(POP / "swap_partners.parquet")
donors = pd.read_parquet(POP / "context_donors.parquet")
partner_of = dict(zip(partners["player"], partners["partner"]))
dedicated_players = np.sort(partners["player"].to_numpy())
dedicated_roles = np.sort(np.unique(val_conflicts.loc[val_conflicts["role"] > 0, "role"].to_numpy()))
if IDENTITY == "player":
    population = val_conflicts[val_conflicts.population == "P-player"].reset_index(drop=True)
else:
    population = val_conflicts[val_conflicts[IDENTITY] > 0].reset_index(drop=True)
index0 = val_conflicts[val_conflicts[IDENTITY] == 0].reset_index(drop=True)
print(f"identity variable: {IDENTITY} | population rows: {len(population):,} | index-0 control rows: {len(index0):,}")
print(f"distinct roles available for the I8 role swap: {len(dedicated_roles)}")

/home/user/code/soccer_mi/model_training_evaluation/src/batching.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  self.tensors[name] = torch.as_tensor(np.asarray(values), device=device)


verified files: 46 | primary seed 103 | identity variable: player
half        discovery  estimation
population                       
P-player         1215         529
index-0          4296        4651
identity variable: player | population rows: 1,744 | index-0 control rows: 8,947
distinct roles available for the I8 role swap: 23


## 1. Terms

| Term | Meaning here |
|---|---|
| **Intervention** | Changing an input (or an activation) on purpose and measuring what changes in the output. |
| **Counterfactual** | The same event with exactly one thing changed — here, who is on the ball. |
| **Target position** | The scoring position of the conflict event in its window. All interventions act there only, unless stated. |
| **KL divergence** | How far the whole 15-action output distribution moved; 0 = unchanged. It catches effects on actions other than a_ctx and a_id. |
| **Norm-matched random vector** | The player's learned vector replaced by a random direction of the same length: a change of the same size that carries no learned player information. |
| **Interaction** | Whether the effect of changing the player depends on the context: zero for a purely additive combination. |

## 2. The interventions

| Code | Change at the target position | Everything else |
|---|---|---|
| I0 | nothing (no-op; must give exactly 0 — a bug detector) | unchanged |
| I1 | identity index → 0, the model's own "no dedicated identity" vector | unchanged |
| I1-all | identity → 0 at every position where the same player acts in the window (variant) | unchanged |
| I2 | player → matched partner (MB02 table: same role, same competition if possible, nearest training count) | team, role, competition and all context unchanged |
| I3 | player → a random other dedicated player (`n_draws` draws) | unchanged |
| I4 | player vector → random direction with the same norm (`n_draws` draws) | unchanged |
| I5 / I6 / I7 | team → 0 / role → 0 / player and team → 0 | unchanged |
| I8 | role → a random other role (`n_draws` draws) | unchanged |

`same_player_as_prev` stays as recorded (Phase 1 decision D3). Every input intervention asserts field by field that only its declared feature at the target changed.

**I8 was added after MB01.** The archived MB01 frozen-reference audit found that role explains 0.769 of the variation
in how far the identity reference moves from the base rate, against 0.122 for the player, and that
94.0% of conflict rows remain conflicts once the player is deleted. The conflict set is therefore
closer to a role-versus-context set than a player-versus-context one. The historical protocol allowed an identity fallback. The new benchmark plan retains `player`; I5-I7 only *remove* those inputs. I8 swaps
the role the way I3 swaps the player, so the decision in Section 7 compares like with like: a
matched swap against a matched swap, both against the same norm-matched random-vector floor (I4).

In [4]:
idplayer_state = torch.load(OUT / "artifacts" / "conflict" / "references" / "id_player_seed0.pt", map_location="cpu", weights_only=False)
idplayer = conflict.IdPlayer(idplayer_state["vocabulary_sizes"])
idplayer.load_state_dict(idplayer_state["state"])
with torch.no_grad():
    all_players = torch.arange(idplayer_state["vocabulary_sizes"]["player"])
    habit_log_probs = torch.log_softmax(idplayer(all_players.unsqueeze(1), torch.zeros(len(all_players), 0))
                                        / idplayer_state["temperature"], dim=-1).numpy()
print("player-only reference table:", habit_log_probs.shape, "| T =", round(idplayer_state["temperature"], 4))

def run_logits(inputs, positions, replace=None):
    logits, _ = hooks.forward(MODEL, inputs, replace=replace)
    return mi_metrics.at_target(logits, positions)

def effect_columns(prefix, base_logits, new_logits, a_ctx, a_id):
    base = base_logits.cpu().numpy(); new = new_logits.cpu().numpy()
    rows = np.arange(len(base))
    base_log = mi_metrics.log_softmax(base); new_log = mi_metrics.log_softmax(new)
    return {f"{prefix}_delta_change": mi_metrics.delta(new, a_ctx, a_id) - mi_metrics.delta(base, a_ctx, a_id),
            f"{prefix}_kl": mi_metrics.kl_divergence(base, new),
            f"{prefix}_dlogp_id": new_log[rows, a_id] - base_log[rows, a_id],
            f"{prefix}_dlogp_ctx": new_log[rows, a_ctx] - base_log[rows, a_ctx]}

def habit_correlation(base_logits, new_logits, player_a, player_b):
    """Pearson r over the 15 actions between the model's change and the player-only reference's change."""
    base = mi_metrics.log_softmax(base_logits.cpu().numpy()); new = mi_metrics.log_softmax(new_logits.cpu().numpy())
    model_change = (new - new.mean(1, keepdims=True)) - (base - base.mean(1, keepdims=True))
    reference_change = habit_log_probs[player_b] - habit_log_probs[player_a]
    reference_change = reference_change - reference_change.mean(1, keepdims=True)
    model_change = model_change - model_change.mean(1, keepdims=True)
    numerator = (model_change * reference_change).sum(1)
    denominator = np.sqrt((model_change ** 2).sum(1) * (reference_change ** 2).sum(1)) + 1e-12
    return numerator / denominator

player-only reference table: (351, 15) | T = 0.9765


In [5]:
@torch.no_grad()
def run_interventions(table, feature, rng_seed):
    generator = torch.Generator().manual_seed(rng_seed)
    rng = np.random.default_rng(rng_seed)
    records = []
    for part, inputs, positions, a_ctx, a_id in mi_setup.event_batches(bundle, val_view, table, BATCH):
        chunk = table.iloc[part]
        a_ctx_np, a_id_np = a_ctx.cpu().numpy(), a_id.cpu().numpy()
        base = run_logits(inputs, positions)
        columns = {"event_row": chunk["event_row"].to_numpy(), "match_id": chunk["match_id"].to_numpy(),
                   "half": chunk["half"].to_numpy(), "delta": mi_metrics.delta(base, a_ctx, a_id)}
        columns.update(effect_columns("I0", base, run_logits(inputs, positions), a_ctx_np, a_id_np))

        removed = interventions.remove_feature(inputs, feature, positions)
        interventions.assert_scope(inputs, removed, [feature], positions)
        columns.update(effect_columns("I1", base, run_logits(removed, positions), a_ctx_np, a_id_np))
        removed_all = interventions.remove_feature_all_occurrences(inputs, feature, positions)
        interventions.assert_scope(inputs, removed_all, [feature])
        columns.update(effect_columns("I1all", base, run_logits(removed_all, positions), a_ctx_np, a_id_np))

        own = chunk[feature].to_numpy()
        if feature == "player":
            partner = np.array([partner_of.get(p, -1) for p in own])
            valid = partner > 0
            swapped = interventions.set_feature(inputs, "player", positions, torch.as_tensor(np.where(valid, partner, own), device=DEVICE))
            interventions.assert_scope(inputs, swapped, ["player"], positions)
            swap_logits = run_logits(swapped, positions)
            effects = effect_columns("I2", base, swap_logits, a_ctx_np, a_id_np)
            for key in effects:
                effects[key] = np.where(valid, effects[key], np.nan)
            columns.update(effects)
            columns["I2_partner"] = partner
            columns["I2_habit_r"] = np.where(valid, habit_correlation(base, swap_logits, own, np.where(valid, partner, own)), np.nan)
            base_log = mi_metrics.log_softmax(base.cpu().numpy()); swap_log = mi_metrics.log_softmax(swap_logits.cpu().numpy())
            change = (swap_log - swap_log.mean(1, keepdims=True)) - (base_log - base_log.mean(1, keepdims=True))
            columns["I2_model_change"] = list(change)
            columns["own_player"] = own

            random_change, random_kl, random_r = [], [], []
            for draw in range(SETTINGS["n_draws"]):
                others = rng.choice(dedicated_players, size=len(own))
                others = np.where(others == own, rng.choice(dedicated_players, size=len(own)), others)
                swapped = interventions.set_feature(inputs, "player", positions, torch.as_tensor(others, device=DEVICE))
                logits = run_logits(swapped, positions)
                effects = effect_columns("I3", base, logits, a_ctx_np, a_id_np)
                random_change.append(np.abs(effects["I3_delta_change"])); random_kl.append(effects["I3_kl"])
                random_r.append(habit_correlation(base, logits, own, others))
            columns["I3_abs_delta_change"] = np.mean(random_change, 0)
            columns["I3_kl"] = np.mean(random_kl, 0)
            columns["I3_habit_r"] = np.mean(random_r, 0)

        vector_change, vector_kl = [], []
        for draw in range(SETTINGS["n_draws"]):
            edit = interventions.norm_matched_vector(None, positions, generator)
            logits = run_logits(inputs, positions, replace={f"factor_{feature}": edit})
            effects = effect_columns("I4", base, logits, a_ctx_np, a_id_np)
            vector_change.append(np.abs(effects["I4_delta_change"])); vector_kl.append(effects["I4_kl"])
        columns["I4_abs_delta_change"] = np.mean(vector_change, 0)
        columns["I4_kl"] = np.mean(vector_kl, 0)

        for code, features_removed in [("I5", ["team"]), ("I6", ["role"]), ("I7", ["player", "team"])]:
            changed = inputs
            for name in features_removed:
                changed = interventions.remove_feature(changed, name, positions)
            interventions.assert_scope(inputs, changed, features_removed, positions)
            columns.update(effect_columns(code, base, run_logits(changed, positions), a_ctx_np, a_id_np))
        own_role = chunk["role"].to_numpy()
        role_change, role_kl = [], []
        for draw in range(SETTINGS["n_draws"]):
            others = rng.choice(dedicated_roles, size=len(own_role))
            others = np.where(others == own_role, rng.choice(dedicated_roles, size=len(own_role)), others)
            swapped_role = interventions.set_feature(inputs, "role", positions, torch.as_tensor(others, device=DEVICE))
            interventions.assert_scope(inputs, swapped_role, ["role"], positions)
            effects = effect_columns("I8", base, run_logits(swapped_role, positions), a_ctx_np, a_id_np)
            role_change.append(np.abs(effects["I8_delta_change"]))
            role_kl.append(effects["I8_kl"])
        columns["I8_abs_delta_change"] = np.mean(role_change, 0)
        columns["I8_kl"] = np.mean(role_kl, 0)
        records.append(pd.DataFrame(columns))
    return pd.concat(records, ignore_index=True)

start = time.time()
effects = run_interventions(population, IDENTITY, 0)
effects["population"] = "identity population"
effects_index0 = run_interventions(index0, IDENTITY, 1)
effects_index0["population"] = "index-0 control"
all_effects = pd.concat([effects, effects_index0], ignore_index=True)
all_effects.drop(columns=["I2_model_change"], errors="ignore").to_parquet(OUT / "results" / "MB04_interventions.parquet", index=False)
print(f"interventions run on {len(effects):,} + {len(effects_index0):,} rows in {time.time() - start:.0f} s")
print("I0 (no-op) largest |delta change|:", float(np.abs(all_effects["I0_delta_change"]).max()), "(must be exactly 0)")
assert float(np.abs(all_effects["I0_delta_change"]).max()) == 0.0

interventions run on 1,744 + 8,947 rows in 136 s
I0 (no-op) largest |delta change|: 0.0 (must be exactly 0)


## 3. E5 — Does the identity input change the output at all? (C2)

**Research question.** Does changing only the player change the output more than an arbitrary change of the same size?

**Hypothesis.** |change in Δ| under the matched swap (I2) exceeds that under the norm-matched random vector (I4).

**Competing outcome.** The two are equal: any change of that size moves the output as much, so the effect is not specific to learned player content.

**Controls and why.** I0 must be exactly 0 (checked above). On index-0 rows I1 is a true no-op, because the player input is already the zero vector. I4 separates "this player's learned vector" from "any vector of that size". The spread of Δ across the five seeds (MB03) is a practical size reference.

**Statistics.** Per-event paired difference |ΔΔ|(I2) − |ΔΔ|(I4) (averaged over draws); match-bootstrap interval and one-sided bootstrap p-value. C2 is decided on the estimation half.

In [6]:
def effect_table(frame, label):
    rows = []
    for code, column in [("I0 no-op", "I0_delta_change"), ("I1 identity -> 0", "I1_delta_change"),
                         ("I1-all every occurrence -> 0", "I1all_delta_change"), ("I2 matched swap", "I2_delta_change"),
                         ("I3 random player (mean |.|)", "I3_abs_delta_change"), ("I4 norm-matched vector (mean |.|)", "I4_abs_delta_change"),
                         ("I5 team -> 0", "I5_delta_change"), ("I6 role -> 0", "I6_delta_change"), ("I7 player+team -> 0", "I7_delta_change"),
                         ("I8 random role (mean |.|)", "I8_abs_delta_change")]:
        if column not in frame:
            continue
        values = np.abs(frame[column].to_numpy())
        keep = ~np.isnan(values)
        if keep.sum() == 0:
            continue
        summary = mi_metrics.summarise(values[keep], frame["match_id"].to_numpy()[keep], N_BOOT)
        kl_column = column.replace("_delta_change", "_kl").replace("_abs", "")
        rows.append({"rows": label, "intervention": code, "n": summary["n"], "mean |change in delta|": summary["mean"],
                     "CI low": summary["ci_low"], "CI high": summary["ci_high"],
                     "mean KL": float(np.nanmean(frame[kl_column])) if kl_column in frame else np.nan})
    return pd.DataFrame(rows)

e5_table = pd.concat([effect_table(effects, "identity population"), effect_table(effects_index0, "index-0 control")])
e5_table.to_csv(OUT / "results" / "MB04_E5_E7_effects.csv", index=False)
print(e5_table.round(4).to_string(index=False))

seed_spread = pd.read_parquet(OUT / "results" / "MB03_event_scores.parquet")
seed_columns = [c for c in seed_spread.columns if c.startswith("delta_seed")]
print(f"\nSize reference: mean over events of the SD of delta across the 5 seeds = {seed_spread[seed_columns].std(axis=1).mean():.4f} nats")

c2_rows = []
if IDENTITY == "player":
    for half in ["estimation", "discovery"]:
        frame = effects[(effects.half == half) & effects["I2_delta_change"].notna()]
        difference = np.abs(frame["I2_delta_change"]) - frame["I4_abs_delta_change"]
        kl_difference = frame["I2_kl"] - frame["I4_kl"]
        summary = mi_metrics.summarise(difference, frame["match_id"], N_BOOT)
        c2_rows.append({"half": half, "n": summary["n"], "mean paired difference |dd|(I2) - |dd|(I4)": summary["mean"],
                        "CI low": summary["ci_low"], "CI high": summary["ci_high"],
                        "one-sided p": mi_metrics.bootstrap_pvalue(difference, frame["match_id"], N_BOOT, alternative="greater"),
                        "mean paired KL difference": float(kl_difference.mean())})
c2_table = pd.DataFrame(c2_rows)
print("\nC2 (decided on the estimation half):")
print(c2_table.round(4).to_string(index=False))

               rows                      intervention    n  mean |change in delta|  CI low  CI high  mean KL
identity population                          I0 no-op 1744                  0.0000  0.0000   0.0000   0.0000
identity population                  I1 identity -> 0 1744                  0.4836  0.4394   0.5312   0.0241
identity population      I1-all every occurrence -> 0 1744                  0.4820  0.4379   0.5289   0.0240
identity population                   I2 matched swap 1717                  0.6302  0.5740   0.6942   0.0406
identity population       I3 random player (mean |.|) 1744                  0.6866  0.6548   0.7182   0.0483
identity population I4 norm-matched vector (mean |.|) 1744                  0.7721  0.7185   0.8339   0.0662
identity population                      I5 team -> 0 1744                  0.3192  0.2782   0.3600   0.0139
identity population                      I6 role -> 0 1744                  0.7026  0.5758   0.8343   0.2283
identity population

In [7]:
fig, axis = plt.subplots(figsize=(10, 4))
series = [("I0", np.abs(effects["I0_delta_change"])), ("I4 vector", effects["I4_abs_delta_change"])]
if IDENTITY == "player":
    series += [("I3 random", effects["I3_abs_delta_change"]), ("I2 matched", np.abs(effects["I2_delta_change"].dropna()))]
series += [("I1 -> 0", np.abs(effects["I1_delta_change"])), ("I1 on index-0", np.abs(effects_index0["I1_delta_change"]))]
axis.boxplot([np.asarray(values) for _, values in series], showfliers=False)
axis.set_xticks(range(1, len(series) + 1)); axis.set_xticklabels([name for name, _ in series])
axis.set_ylabel("|change in delta| (nats)"); axis.set_title("How far each intervention moves the arbitration")
plt.tight_layout(); plt.savefig(OUT / "figures" / "MB04_E5_effects.png", bbox_inches="tight"); plt.show()

/tmp/ipykernel_300635/3181799842.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / "figures" / "MB04_E5_effects.png", bbox_inches="tight"); plt.show()


**How to read the figure.** Each box is the distribution over events of how far an intervention moved Δ. The identity signal is the gap between the "I2 matched" box and the "I4 vector" box; "I1 on index-0" must sit at exactly zero.

**Researcher interpretation** *(written by the researcher after reading the numbers above; the code never writes these words)*

- What was tested:
- Why it was tested:
- What was observed:
- What the result supports:
- What it contradicts:
- Alternative explanations that remain:
- What it does not establish:
- Evidence label (Observed / Supported / Strong causal evidence / Hypothesized):

## 4. E6 — When identity matters, does the model use the player's habits? (C3)

For a swap A → B, the model's change in centred log-probabilities over the 15 actions is correlated (Pearson r) with the player-only reference's change log π_B − log π_A. A positive mean r means the model moves in the direction of the new player's habits.

**Hypothesis.** Mean r > 0 for matched swaps (C3, estimation half). **Competing outcome.** r ≈ 0: the player vector changes outputs in ways unrelated to habits.

**Control.** The same model change correlated with the habit change of a *random other* player (`n_permutations` permutations) gives the null distribution: if the model tracked habits, the true partner's habits should fit its change better than a random player's.

In [8]:
habit_rows = []
if IDENTITY == "player":
    frame = effects[effects["I2_habit_r"].notna()].reset_index(drop=True)
    rng = np.random.default_rng(0)
    model_change = np.stack(frame["I2_model_change"].to_numpy())
    own_players = frame["own_player"].to_numpy()

    def mean_r(reference_players, rows):
        reference = habit_log_probs[reference_players] - habit_log_probs[own_players[rows]]
        reference = reference - reference.mean(1, keepdims=True)
        change = model_change[rows] - model_change[rows].mean(1, keepdims=True)
        r = (change * reference).sum(1) / (np.sqrt((change ** 2).sum(1) * (reference ** 2).sum(1)) + 1e-12)
        return float(r.mean())

    for half in ["estimation", "discovery"]:
        rows = np.where(frame.half == half)[0]
        summary = mi_metrics.summarise(frame.loc[rows, "I2_habit_r"], frame.loc[rows, "match_id"], N_BOOT)
        null = [mean_r(rng.choice(dedicated_players, size=len(rows)), rows) for _ in range(SETTINGS["n_permutations"])]
        habit_rows.append({"half": half, "n": summary["n"], "mean r (matched swap)": summary["mean"],
                           "CI low": summary["ci_low"], "CI high": summary["ci_high"],
                           "null mean r (random reference player)": float(np.mean(null)),
                           "null 95th percentile": float(np.percentile(null, 95)),
                           "mean r (random-player swap, I3)": float(frame.loc[rows, "I3_habit_r"].mean())})
habit_table = pd.DataFrame(habit_rows)
print(habit_table.round(4).to_string(index=False))

      half    n  mean r (matched swap)  CI low  CI high  null mean r (random reference player)  null 95th percentile  mean r (random-player swap, I3)
estimation  529                 0.0758  0.0187   0.1301                                 0.0344                0.0487                           0.0713
 discovery 1188                 0.0934  0.0387   0.1405                                 0.0503                0.0603                           0.0720


The random-player swap (I3) serves as a second reading of the same question with larger identity changes; its r is reported next to the matched swap's.

**Researcher interpretation** *(written by the researcher after reading the numbers above; the code never writes these words)*

- What was tested:
- Why it was tested:
- What was observed:
- What the result supports:
- What it contradicts:
- Alternative explanations that remain:
- What it does not establish:
- Evidence label (Observed / Supported / Strong causal evidence / Hypothesized):

## 5. E7 — Do team and role carry the identity effect instead?

The E5 table above already contains I5 (team → 0), I6 (role → 0) and I7 (player and team → 0). This matters because model_training_evaluation found that removing player and team *improved* prediction: the model may nonetheless use them. The table below compares each with I1.

In [9]:
e7 = e5_table[e5_table.intervention.str.startswith(("I1 ", "I5", "I6", "I7", "I8"))]
print(e7.round(4).to_string(index=False))

               rows              intervention    n  mean |change in delta|  CI low  CI high  mean KL
identity population          I1 identity -> 0 1744                  0.4836  0.4394   0.5312   0.0241
identity population              I5 team -> 0 1744                  0.3192  0.2782   0.3600   0.0139
identity population              I6 role -> 0 1744                  0.7026  0.5758   0.8343   0.2283
identity population       I7 player+team -> 0 1744                  0.5558  0.5222   0.5899   0.0318
identity population I8 random role (mean |.|) 1744                  0.6718  0.5605   0.8002   0.3057
    index-0 control          I1 identity -> 0 8947                  0.0000  0.0000   0.0000   0.0000
    index-0 control              I5 team -> 0 8947                  0.2980  0.2547   0.3443   0.0100
    index-0 control              I6 role -> 0 8947                  0.9223  0.8544   0.9977   0.4157
    index-0 control       I7 player+team -> 0 8947                  0.2980  0.2547   0.3443

**Researcher interpretation** *(written by the researcher after reading the numbers above; the code never writes these words)*

- What was tested:
- Why it was tested:
- What was observed:
- What the result supports:
- What it contradicts:
- Alternative explanations that remain:
- What it does not establish:
- Evidence label (Observed / Supported / Strong causal evidence / Hypothesized):

## 6. E8 — Additive or gated? (C4)

**Design.** For each P-player row with a partner and a context donor: players A (the event's own) and B (its partner) × contexts X (the event's own window) and Y (the donor window: another P-player event from a different match, same role and same scoring position). The player of each window's target is set to A or B; team, role and competition stay with the context (decision D7). The action pair (a_ctx, a_id) of X is used in all four cells.

```latex
I = m(A,X) - m(A,Y) - m(B,X) + m(B,Y)
```

**Floor.** The same four cells recomputed with every attention and MLP output frozen to its value in the A-run of that context, so the player swap can reach the logits only through the direct path; what remains of I then comes from the final LayerNorm alone.

**Hypotheses.** H1 (additive): mean |I| ≈ mean |I_floor|. H2 (gated): mean |I| > mean |I_floor|. **Test (C4):** paired match-bootstrap interval of |I| − |I_floor| on the estimation half.

In [10]:
factorial_rows = []
if IDENTITY == "player":
    units = population.merge(donors, on="event_row")
    units["partner"] = [partner_of.get(p, -1) for p in units["player"]]
    units = units[units["partner"] > 0].reset_index(drop=True)
    frozen_sites = [f"attn_out_{l}" for l in range(N_LAYERS)] + [f"mlp_out_{l}" for l in range(N_LAYERS)]
    with torch.no_grad():
        for part in mi_setup.batch_slices(len(units), BATCH):
            chunk = units.iloc[part]
            x_inputs, _ = mi_setup.window_inputs(bundle, val_view, chunk["window"].to_numpy())
            y_inputs, _ = mi_setup.window_inputs(bundle, val_view, chunk["donor_window"].to_numpy())
            positions = torch.as_tensor(chunk["position"].to_numpy(), device=DEVICE)
            a_ctx = torch.as_tensor(chunk["a_ctx"].to_numpy(), device=DEVICE)
            a_id = torch.as_tensor(chunk["a_id"].to_numpy(), device=DEVICE)
            player_a = torch.as_tensor(chunk["player"].to_numpy(), device=DEVICE)
            player_b = torch.as_tensor(chunk["partner"].to_numpy(), device=DEVICE)
            cells = {}
            floor = {}
            for context_name, context_inputs in [("X", x_inputs), ("Y", y_inputs)]:
                with_a = interventions.set_feature(context_inputs, "player", positions, player_a)
                with_b = interventions.set_feature(context_inputs, "player", positions, player_b)
                logits_a, cache_a = hooks.forward(MODEL, with_a, capture=frozen_sites)
                logits_b, _ = hooks.forward(MODEL, with_b)
                frozen = {site: (lambda value, clean=cache_a[site]: clean) for site in frozen_sites}
                logits_b_direct, _ = hooks.forward(MODEL, with_b, replace=frozen)
                cells[("A", context_name)] = mi_metrics.delta(mi_metrics.at_target(logits_a, positions), a_ctx, a_id)
                cells[("B", context_name)] = mi_metrics.delta(mi_metrics.at_target(logits_b, positions), a_ctx, a_id)
                floor[("A", context_name)] = cells[("A", context_name)]
                floor[("B", context_name)] = mi_metrics.delta(mi_metrics.at_target(logits_b_direct, positions), a_ctx, a_id)
            interaction = cells[("A", "X")] - cells[("A", "Y")] - cells[("B", "X")] + cells[("B", "Y")]
            interaction_floor = floor[("A", "X")] - floor[("A", "Y")] - floor[("B", "X")] + floor[("B", "Y")]
            factorial_rows.append(pd.DataFrame({"event_row": chunk["event_row"].to_numpy(), "match_id": chunk["match_id"].to_numpy(),
                                                "half": chunk["half"].to_numpy(), "rule": chunk["rule"].to_numpy(),
                                                "m_AX": cells[("A", "X")], "m_AY": cells[("A", "Y")],
                                                "m_BX": cells[("B", "X")], "m_BY": cells[("B", "Y")],
                                                "interaction": interaction, "interaction_floor": interaction_floor}))
factorial = pd.concat(factorial_rows, ignore_index=True) if factorial_rows else pd.DataFrame()
if len(factorial):
    factorial.to_parquet(OUT / "results" / "MB04_factorial.parquet", index=False)
    c4_rows = []
    for half in ["estimation", "discovery"]:
        frame = factorial[factorial.half == half]
        difference = np.abs(frame["interaction"]) - np.abs(frame["interaction_floor"])
        summary = mi_metrics.summarise(difference, frame["match_id"], N_BOOT)
        c4_rows.append({"half": half, "units": summary["n"], "mean |I|": float(np.abs(frame["interaction"]).mean()),
                        "mean |I_floor|": float(np.abs(frame["interaction_floor"]).mean()),
                        "mean paired difference": summary["mean"], "CI low": summary["ci_low"], "CI high": summary["ci_high"],
                        "one-sided p": mi_metrics.bootstrap_pvalue(difference, frame["match_id"], N_BOOT, alternative="greater")})
    c4_table = pd.DataFrame(c4_rows)
    print(c4_table.round(4).to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    means = factorial[["m_AX", "m_AY", "m_BX", "m_BY"]].mean()
    axes[0].plot(["X (own context)", "Y (donor context)"], [means["m_AX"], means["m_AY"]], "o-", label="player A (own)")
    axes[0].plot(["X (own context)", "Y (donor context)"], [means["m_BX"], means["m_BY"]], "o-", label="player B (partner)")
    axes[0].set_ylabel("mean delta (nats)"); axes[0].legend(); axes[0].set_title("Interaction plot (parallel lines = additive)")
    axes[1].hist(np.abs(factorial["interaction"]), bins=40, alpha=0.6, label="|I|")
    axes[1].hist(np.abs(factorial["interaction_floor"]), bins=40, alpha=0.6, label="|I floor| (direct path only)")
    axes[1].set_xlabel("nats"); axes[1].legend(); axes[1].set_title("Interaction against its LayerNorm floor")
    plt.tight_layout(); plt.savefig(OUT / "figures" / "MB04_factorial.png", bbox_inches="tight"); plt.show()
else:
    c4_table = pd.DataFrame()
    print("factorial not run (identity variable is not the player, or no units)")

      half  units  mean |I|  mean |I_floor|  mean paired difference  CI low  CI high  one-sided p
estimation    529    0.8307          0.0438                  0.7869  0.7171   0.8793       0.0005
 discovery   1187    0.7683          0.0415                  0.7268  0.6472   0.8361       0.0005


/tmp/ipykernel_300635/2097722628.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / "figures" / "MB04_factorial.png", bbox_inches="tight"); plt.show()


**Researcher interpretation** *(written by the researcher after reading the numbers above; the code never writes these words)*

- What was tested:
- Why it was tested:
- What was observed:
- What the result supports:
- What it contradicts:
- Alternative explanations that remain:
- What it does not establish:
- Evidence label (Observed / Supported / Strong causal evidence / Hypothesized):

## 7. Decision point: which identity variable do MB05–MB11 study?

The numbers that decide it are printed together below. **The decision is the researcher's**, made by reading them, and is recorded by editing `identity_variable` in `config/mi_config.json` (to `"team"` or `"role"`) before MB05 runs; the code does not choose. The pre-registered rule: if the matched-swap effect does not exceed the norm-matched random-vector effect, fall back to team, then role.

In [11]:
print("C2 (player identity specificity):")
print(c2_table.round(4).to_string(index=False) if len(c2_table) else "  not applicable")
print("\nTeam and role (E7), identity population:")
print(e5_table[(e5_table["rows"] == "identity population") & e5_table.intervention.str.startswith(("I1 ", "I5", "I6", "I8"))].round(4).to_string(index=False))
print("\nCurrent config/mi_config.json:", mi_setup.load_json(OUT / "config" / "mi_config.json"))

summary = {"identity_variable": IDENTITY, "E5": e5_table.to_dict(orient="records"), "C2": c2_table.to_dict(orient="records"),
           "C3": habit_table.to_dict(orient="records"), "C4": c4_table.to_dict(orient="records"), "run_mode": RUN_MODE}
mi_setup.save_json(OUT / "results" / "MB04_summary.json", summary)
fingerprint.record_files(MANIFEST, {"results/MB04_interventions.parquet": OUT / "results" / "MB04_interventions.parquet",
                                    "results/MB04_summary.json": OUT / "results" / "MB04_summary.json"})
fingerprint.register(REGISTRY, {"notebook": "MB04", "experiment": "E5-E8 counterfactuals", "identity_variable": IDENTITY,
                                "n": len(effects), "seed": PRIMARY_SEED, "run_mode": RUN_MODE})

C2 (player identity specificity):
      half    n  mean paired difference |dd|(I2) - |dd|(I4)  CI low  CI high  one-sided p  mean paired KL difference
estimation  529                                     -0.0882 -0.1734   0.0288       0.9445                    -0.0256
 discovery 1188                                     -0.1687 -0.2247  -0.1148       1.0000                    -0.0262

Team and role (E7), identity population:
               rows              intervention    n  mean |change in delta|  CI low  CI high  mean KL
identity population          I1 identity -> 0 1744                  0.4836  0.4394   0.5312   0.0241
identity population              I5 team -> 0 1744                  0.3192  0.2782   0.3600   0.0139
identity population              I6 role -> 0 1744                  0.7026  0.5758   0.8343   0.2283
identity population I8 random role (mean |.|) 1744                  0.6718  0.5605   0.8002   0.3057

Current config/mi_config.json: {'identity_variable': 'player', 'pri

**Analysis decision.** This new benchmark analysis retains `player` as its identity variable regardless of C2, as recorded before these measurements in `analysis_plan.json`. Team and role comparisons remain exploratory diagnostics; there is no post-result identity fallback.

## R. Denominators and swap coherence (REPAIR 2026-10-04; F12, F13)

Descriptive diagnostics computed from the data and the saved intervention table; no model is run. A swap is called
**coherent** here only if (a) the partner was matched within the same competition, (b) the partner's most frequent training team
equals the event's team, and (c) `same_player_as_prev` is 0 at the target, so the swap does not contradict the history flag.
This definition was written after the results and is labelled post-hoc. It splits C2 descriptively and is not a new test.

In [12]:
features = {name: tensor.cpu().numpy() for name, tensor in bundle["encoded"].tensors.items()}
train_rows = np.where(events["split"].to_numpy() == "train")[0]
train = pd.DataFrame({"player": features["player"][train_rows], "team": features["team"][train_rows]})
modal_team = train[train.player > 0].groupby("player")["team"].agg(lambda s: s.value_counts().index[0])

denominators = pd.DataFrame([
    {"population": "P-player (dedicated player vector)", "rows": len(population), "estimation": int((population.half == "estimation").sum()),
     "discovery": int((population.half == "discovery").sum())},
    {"population": "P-player with a swap partner (C2, C3)", "rows": int(effects["I2_delta_change"].notna().sum()),
     "estimation": int(effects.loc[effects.half == "estimation", "I2_delta_change"].notna().sum()),
     "discovery": int(effects.loc[effects.half == "discovery", "I2_delta_change"].notna().sum())},
    {"population": "index-0 (no dedicated player vector)", "rows": len(index0), "estimation": int((index0.half == "estimation").sum()),
     "discovery": int((index0.half == "discovery").sum())},
    {"population": "matched non-conflict controls", "rows": len(controls), "estimation": int((controls.half == "estimation").sum()) if "half" in controls else np.nan,
     "discovery": int((controls.half == "discovery").sum()) if "half" in controls else np.nan}])
print(denominators.to_string(index=False))

rule_of = dict(zip(partners["player"], partners["rule"]))
coherence = effects[effects["I2_delta_change"].notna()][["event_row", "match_id", "half", "own_player", "I2_partner", "I2_delta_change", "I4_abs_delta_change"]].copy()
coherence["team"] = features["team"][coherence["event_row"].to_numpy()]
coherence["partner modal team"] = coherence["I2_partner"].map(modal_team)
coherence["same competition rule"] = coherence["own_player"].map(rule_of).eq("same role and competition")
coherence["partner same team"] = coherence["partner modal team"].to_numpy() == coherence["team"].to_numpy()
coherence["history flag consistent"] = features["same_player_as_prev"][coherence["event_row"].to_numpy()] == 0
coherence["coherent (post-hoc)"] = coherence["same competition rule"] & coherence["partner same team"] & coherence["history flag consistent"]
shares = coherence[["same competition rule", "partner same team", "history flag consistent", "coherent (post-hoc)"]].mean()
print("\nshare of matched swaps with each property:"); print(shares.round(3).to_string())
rows = []
for half in ["estimation", "discovery"]:
    for coherent in [True, False]:
        block = coherence[(coherence.half == half) & (coherence["coherent (post-hoc)"] == coherent)]
        if len(block) < 10:
            continue
        summary = mi_metrics.summarise(np.abs(block["I2_delta_change"]) - block["I4_abs_delta_change"], block["match_id"], N_BOOT)
        rows.append({"half": half, "coherent swap (post-hoc)": coherent, "n": summary["n"], "matches": summary["n_matches"],
                     "mean |dd|(I2) - |dd|(I4)": summary["mean"], "CI low": summary["ci_low"], "CI high": summary["ci_high"], "uncertainty": "descriptive only: one match" if summary["n_matches"] < 2 else "match bootstrap"})
c2_by_coherence = pd.DataFrame(rows)
print("\nC2 split by swap coherence (descriptive, post-hoc):"); print(c2_by_coherence.round(4).to_string(index=False))
roles_pool = dedicated_roles
print(f"\nI8 random role: expected share of draws left unchanged by the 'draw, redraw once' rule "
      f"(uniform over {len(roles_pool)} roles) = {1 / len(roles_pool) ** 2:.4f}; I3 random player: {1 / len(dedicated_players) ** 2:.6f}. "
      "Both are small; reported, not corrected (F15 pattern).")
denominators.to_csv(OUT / "results" / "MB04_denominators_corrected.csv", index=False)
coherence.to_parquet(OUT / "results" / "MB04_swap_coherence_corrected.parquet", index=False)
c2_by_coherence.to_csv(OUT / "results" / "MB04_C2_by_coherence_corrected.csv", index=False)

                           population  rows  estimation  discovery
   P-player (dedicated player vector)  1744         529       1215
P-player with a swap partner (C2, C3)  1717         529       1188
 index-0 (no dedicated player vector)  8947        4651       4296
        matched non-conflict controls   252          65        187

share of matched swaps with each property:
same competition rule      0.839
partner same team          0.227
history flag consistent    0.753
coherent (post-hoc)        0.175

C2 split by swap coherence (descriptive, post-hoc):
      half  coherent swap (post-hoc)   n  matches  mean |dd|(I2) - |dd|(I4)  CI low  CI high                 uncertainty
estimation                      True  16        1                   -0.4067     NaN      NaN descriptive only: one match
estimation                     False 513       12                   -0.0783 -0.1617   0.0313             match bootstrap
 discovery                      True 285        5                   -0.13

**Reading.** The coherent-swap subset is a limited post hoc proxy for plausible combinations. A one-match subset has no match-bootstrap confidence interval and is descriptive only; it cannot establish that C2 is robust to implausible combinations. The split does not replace the full-population C2 estimate.